### Gold Layer Overview - Formula 1 Data Lakehouse

The **gold layer** is the final stage of our **medallion architecture** (bronze -> silver -> gold). It contains clean, business-ready **dimension and fact tables** optimized for analytics, dashboards, and reporting. All tables are written as Delta format in `formula1.gold`.

---

#### Pipeline Steps

##### Step 1: Build Nationality Reference Table
**Notebook:** **`11) Building Nationality Reference table`**  
**Target:** `formula1.gold.ref_nationalaty_regions`

A static lookup table that maps **41 unique nationalities** (found across drivers and constructors) to **6 geographic regions**: Europe, Asia, South America, North America, Oceania, and Africa. This table is manually defined and used by both dimension tables below to enrich data with regional information.

---

##### Step 2: Build Races Dimension
**Notebook:** **`01) Build Races Dimention`**  
**Target:** `formula1.gold.dim_races`

| Source Tables | Join Key | Output Columns |
| --- | --- | --- |
| `silver.races` + `silver.circuits` | `circuit_id` (inner join) | season, round, race_name, race_date, circuit_name, locality, country |

Combines race event details with circuit location into a single denormalized dimension - no joins needed at query time.

---

##### Step 3: Build Constructors Dimension
**Notebook:** **`02) Build Constructors Dimention`**  
**Target:** `formula1.gold.dim_constructors`

| Source Tables | Join Key | Output Columns |
| --- | --- | --- |
| `silver.constructors` + `gold.ref_nationalaty_regions` | `nationality` (left outer join) | constructor_id, constructor_name, nationality, nationality_region |

Enriches each constructor/team with its geographic region. Left outer join ensures all constructors are kept even without a region match.

---

##### Step 4: Build Drivers Dimension
**Notebook:** **`03) Build Drivers Dimention`**  
**Target:** `formula1.gold.dim_drivers`

| Source Tables | Join Key | Output Columns |
| --- | --- | --- |
| `silver.drivers` + `gold.ref_nationalaty_regions` | `nationality` (left outer join) | driver_id, driver_name, date_of_birth, nationality, nationality_region |

Enriches each driver with geographic region information for regional analysis.

---

##### Step 5: Build Session Results Fact Table
**Notebook:** **`04 Build Session Results Fact`**  
**Target:** `formula1.gold.facts_session_results`

The central **fact table** that unifies race and sprint results:
1. Reads `silver.results` (adds `session_type = 'RACE'`) and `silver.sprints` (adds `session_type = 'SPRINT'`)
2. Unions both into a single DataFrame using `unionByName()`
3. Derives analytical boolean columns:

| Derived Column | Logic | Purpose |
| --- | --- | --- |
| `is_win` | `final_position == 1` | Count wins with `SUM(is_win)` |
| `is_podium` | `final_position BETWEEN 1 AND 3` | Filter/count podium finishes |
| `has_points` | `points > 0` | Identify points-scoring sessions |

---

#### Final Gold Schema - Entity Relationship Diagram

---

#### Summary of Gold Tables

| Table | Type | Key Columns |
| --- | --- | --- |
| `ref_nationalaty_regions` | Reference | nationality, region |
| `dim_races` | Dimension | season, round, race_name, circuit_name, country |
| `dim_constructors` | Dimension | constructor_id, constructor_name, nationality_region |
| `dim_drivers` | Dimension | driver_id, driver_name, nationality_region |
| `facts_session_results` | Fact | season, round, driver_id, constructor_id, points, is_win, is_podium |

In [0]:
displayHTML('<iframe style="border: 1px solid rgba(0, 0, 0, 0.1);" width="800" height="450" src="https://embed.figma.com/board/sqzbUFRL8caRHqFrJgYmxi/Formula1-Gold-Schema-ERD---Matching-Colors?node-id=0-1&embed-host=share" allowfullscreen></iframe>')